# Preprocessing QA report

## Goal

Produce a compact, machine-readable QA summary for v1 or v2 without
modifying the source parquet. Diagnostic MC values are never treated as
reconstructed mother targets.

## Setup and data

In [ ]:
from pathlib import Path
import json, os, sys
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists(): REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))
from hypertagging.data.notebook_fixtures import write_notebook_fixture_v4
from hypertagging.preprocessing.schema_v4 import load_payload_v4

requested = os.environ.get("HYPERTAGGING_PARQUET", "").strip()
FIXTURE_MODE = not bool(requested)
INPUT_PATH = Path(requested) if requested else Path("/tmp/hypertagging_notebook_fixture_v3.parquet")
if FIXTURE_MODE: write_notebook_fixture_v4(INPUT_PATH)
if not INPUT_PATH.exists(): raise FileNotFoundError(INPUT_PATH)
FIGURE_DIR = Path(os.environ.get("HYPERTAGGING_FIGURE_DIR", "/tmp/hypertagging_figures/qa"))
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
QA_JSON = Path(os.environ.get("HYPERTAGGING_QA_JSON", str(FIGURE_DIR / "preprocessing_qa.json")))
payload = load_payload_v4(INPUT_PATH)
events = payload["events"]
print("TINY FIXTURE QA — NOT REAL DATA" if FIXTURE_MODE else "REAL-DATA SAMPLE QA")

## Results

In [ ]:
level_violations, closure_residuals, invalid_values = [], [], 0
for event in events:
    by_id = {int(node["node_id"]): node for node in event["nodes"]}
    for mother in event["nodes"]:
        common = [mother[name] for name in ("px", "py", "pz", "energy", "mass", "charge")]
        invalid_values += int(not np.isfinite(common).all())
        daughters = [by_id[int(child)] for child in mother["daughter_ids"]]
        if not daughters: continue
        expected_level = 1 + max(daughter["level"] for daughter in daughters)
        if mother["level"] != expected_level:
            level_violations.append([event["event_uid"], mother["node_id"], mother["level"], expected_level])
        closure_residuals.append([
            mother[name] - sum(daughter[name] for daughter in daughters)
            for name in ("energy", "px", "py", "pz")
        ])
closure = np.asarray(closure_residuals, dtype=float)
maximum_closure = float(np.abs(closure).max()) if closure.size else 0.0
event_uids = [event["event_uid"] for event in events]
all_nodes = [node for event in events for node in event["nodes"]]
token_range_pass = all(
    0 <= int(node["input_pid_token"]) < 41
    and 0 <= int(node["pid_target_token"]) < 41
    for node in all_nodes
)
no_truth_leakage_contract_pass = all(
    node["leaf_kinematics_mode"] != "raw_track_predicted_pid"
    or (
        int(node["input_pid_token"]) == 0
        and node["energy_source"].startswith("canonical_")
    )
    for node in all_nodes
)
node_kind_pass = all(
    node["node_kind"] in {"unknown","track","ecl_cluster","composite","other"}
    for node in all_nodes
)
partial_rate = float(np.mean([node["partial_missing_daughters"] for node in all_nodes]))
query_counts = [
    sum(node["level"] == level for node in event["nodes"])
    for event in events
    for level in set(node["level"] for node in event["nodes"] if node["level"] > 0)
]
cardinalities = [len(node["daughter_ids"]) for node in all_nodes if node["daughter_ids"]]
report = {
    "mode": "fixture" if FIXTURE_MODE else "real_sample",
    "schema_version": payload["schema_version"],
    "events": len(events),
    "unique_event_uids": len(set(event_uids)),
    "duplicate_event_uids": len(event_uids) - len(set(event_uids)),
    "nodes": sum(len(event["nodes"]) for event in events),
    "level_violations": level_violations,
    "invalid_common_value_nodes": invalid_values,
    "maximum_absolute_p4_closure_residual": maximum_closure,
    "p4_closure_pass": maximum_closure < 1e-8,
    "no_truth_leakage_pass": no_truth_leakage_contract_pass,
    "token_range_pass": token_range_pass,
    "node_kind_consistency_pass": node_kind_pass,
    "pid_likelihood_availability_explicit": all(
        "pid_likelihood_availability" in node for node in all_nodes
    ),
    "partial_decay_rate": partial_rate,
    "query_overflow_rate_at_capacity_8": float(np.mean([value > 8 for value in query_counts])) if query_counts else 0.0,
    "cardinality_overflow_rate_at_capacity_6": float(np.mean([value > 6 for value in cardinalities])) if cardinalities else 0.0,
    "schema_config_consistency_pass": payload["schema_version"] == "direct-mdst-tree-v4",
    "composite_pid_input_truth_separated_pass": all(
        "daughter_input_pid_histogram" in node
        and "daughter_truth_pid_histogram" in node
        and "daughter_pid_histogram" not in node
        for node in all_nodes
    ),
    "recursive_completeness_available": all(
        "recursive_reconstructable_complete" in node
        for node in all_nodes
    ),
}
QA_JSON.parent.mkdir(parents=True, exist_ok=True)
QA_JSON.write_text(json.dumps(report, indent=2, sort_keys=True), encoding="utf-8")
print(json.dumps(report, indent=2))
if closure.size:
    fig, axes = plt.subplots(2, 2, figsize=(10, 7))
    for axis, values, label in zip(axes.flat, closure.T, ["delta E", "delta px", "delta py", "delta pz"]):
        axis.hist(values, bins=25); axis.set_title(label)
    fig.tight_layout(); fig.savefig(FIGURE_DIR / "qa_p4_closure.png"); plt.show()
assert not level_violations
assert invalid_values == 0
assert maximum_closure < 1e-8
assert token_range_pass and no_truth_leakage_contract_pass and node_kind_pass
print("Machine-readable summary:", QA_JSON)

## Takeaways

A passing report verifies finite common values, exact retained-level
recurrence, unique event IDs, and daughter-summed composite p4 closure.
It does not validate full-training physics performance.